In [ ]:
# GPT-2-Large (774M) on WikiText-103: third scaling point (124M, 355M already
# committed from earlier runs). Only computes the new large model here, then
# regenerates the combined 3-way scaling comparison using all three.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone -q https://github.com/abdurrahmanrussel/QAT-VQ-Compression.git repo
%cd repo
!git checkout -q gpt2medium-wikitext103-qatvq
!git log --oneline -3
!ls gpt2_scaling/artifacts/gpt2/results.json gpt2_scaling/artifacts/gpt2-medium/results.json

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.18" scikit-learn matplotlib bitsandbytes

In [ ]:
%cd /kaggle/working/repo/gpt2_scaling
import os
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
try:
    import bitsandbytes as bnb
    print("bitsandbytes OK, version", bnb.__version__)
except Exception as e:
    print("bitsandbytes NOT available, will fall back to plain AdamW (may OOM on gpt2-large):", e)

## GPT-2-Large (774M) on WikiText-103
~2.2x GPT-2-Medium's params -> smaller batches (16GB VRAM budget). Uses the
SAME 20M-char calibration subset as the 124M/355M runs this time (the first
successful run used 10M chars to bound runtime, which turned out to
confound the quality-vs-scale comparison -- this rerun removes that
confound at the cost of a longer session, ~18h estimated vs ~9h before).
Needs 8-bit AdamW (bitsandbytes) even at bs=1 -- plain AdamW's fp32
optimizer state alone (~12.4GB for 774M params) saturates a T4 regardless
of batch size, proven by two earlier OOM failures at bs=2 and bs=1.

In [ ]:
!python train_baseline.py --model gpt2-large --epochs 1 --bs 1 --grad_accum 8 --train_subset_chars 20000000

In [ ]:
!python run_experiments.py --model gpt2-large --sub_dim 2 --K 256 --qat_epochs 1 --ft_lr 1e-5 --seeds 0 1 2 --bs_train 1 --bs_eval 1 --train_subset_chars 20000000

In [ ]:
# Check the seed search output above for the actual best seed on THIS run before trusting the default below.
!python finetune_vq.py --model gpt2-large --seed 1 --epochs 2 --lr 5e-6 --bs_train 1 --bs_eval 1 --train_subset_chars 20000000

In [ ]:
# Regenerates per-model figures for gpt2-large AND the combined 3-way scaling
# comparison (make_figures.py reads all three MODELS dirs, two of which came
# from git, one freshly computed above).
!python make_figures.py
!cat artifacts/scaling_table.md

In [ ]:
# Push results back to GitHub. Requires a Kaggle Secret named GITHUB_TOKEN
# (fine-grained PAT, repo=QAT-VQ-Compression, Contents: Read and write) --
# add it via this kernel's editor -> Add-ons -> Secrets before running.
# If this fails (Kaggle's internal secrets service has been flaky on every
# run so far), results are still recoverable via `kaggle kernels output`.
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

import subprocess
def sh(cmd):
    print('$', cmd.replace(token, '***') if token in cmd else cmd)
    subprocess.run(cmd, shell=True, check=True)

sh('git config user.email "abdurrahmanrussel77@gmail.com"')
sh('git config user.name "Md Abdur Rahman"')
sh('git add artifacts/gpt2-large/results.json artifacts/gpt2-large/results_table.md '
   'artifacts/gpt2-large/figures/ artifacts/scaling_table.md artifacts/figures/scaling_comparison.png')
sh('git commit -m "Add GPT-2-Large (774M) scaling point from Kaggle" || echo "nothing to commit"')
sh(f'git push https://{token}@github.com/abdurrahmanrussel/QAT-VQ-Compression.git gpt2medium-wikitext103-qatvq')